# Text-to-SQL: ask questions of the gold tables in plain English
A chat model writes a SQL query from a question. The model sees only table names, column names, and allowed values, never the data. A guard allows only a single read-only SELECT on gold tables, and the query runs in Spark so the numbers come from the database.

## 1. Read the gold tables and columns
Reads every gold table, column, and data type from the catalog into text for the model. `email_hash` and the RAG and lookup objects (`comment_enrichment`, `comment_embeddings`, `comment_index`) are left out.

In [0]:
MODEL = "databricks-meta-llama-3-3-70b-instruct"  # Databricks-hosted chat model; if this errors, pick a name from Serving

cols = spark.sql("""
SELECT table_name, column_name, data_type
FROM zomato_ai.information_schema.columns
WHERE table_schema = 'gold'
  AND column_name <> 'email_hash'
  AND table_name NOT IN ('comment_enrichment', 'comment_embeddings', 'comment_index')
ORDER BY table_name, ordinal_position
""").collect()

schema = {}
for r in cols:
    schema.setdefault(r.table_name, []).append(f"{r.column_name} ({r.data_type})")

schema_text = "\n".join(f"zomato_ai.gold.{t}: " + ", ".join(c) for t, c in schema.items())
print(schema_text)

zomato_ai.gold.agg_city_order_status: restaurant_city (STRING), order_status (STRING), orders (LONG), share_of_city_orders (DOUBLE)
zomato_ai.gold.agg_daily_sales: order_date (DATE), total_orders (LONG), delivered_orders (LONG), cancelled_orders (LONG), refunded_orders (LONG), revenue (DECIMAL), avg_delivery_time_min (DOUBLE), avg_customer_rating (DOUBLE)
zomato_ai.gold.agg_restaurant_performance: restaurant_id (STRING), restaurant_name (STRING), city (STRING), total_orders (LONG), delivered_orders (LONG), cancel_rate (DOUBLE), revenue (DECIMAL), avg_order_value (DECIMAL), avg_delivery_time_min (DOUBLE), avg_customer_rating (DOUBLE)
zomato_ai.gold.agg_restaurant_sentiment: restaurant_id (STRING), restaurant_name (STRING), city (STRING), total_reviews (LONG), avg_rating (DOUBLE), positive_reviews (LONG), neutral_reviews (LONG), mixed_reviews (LONG), negative_reviews (LONG), pct_negative (DOUBLE), top_complaint_topic (STRING), top_complaint_count (LONG)
zomato_ai.gold.dim_food: food_id (

## 2. Find the exact values in category columns
Lists the stored values of every text column with 20 or fewer distinct values (order status, payment method, sentiment, and so on), so the model spells filters exactly as stored.

In [0]:
from collections import defaultdict

string_cols = defaultdict(list)
for r in cols:
    if r.data_type == "STRING" and r.table_name != "comment_enrichment":
        string_cols[r.table_name].append(r.column_name)

MAX_VALUES = 20
value_lines = []
for t, cs in string_cols.items():
    counts = spark.sql(
        "SELECT " + ", ".join(f"approx_count_distinct(`{c}`) AS `{c}`" for c in cs) +
        f" FROM zomato_ai.gold.{t}"
    ).collect()[0]
    for c in cs:
        if counts[c] <= MAX_VALUES:
            vals = [r[0] for r in spark.sql(
                f"SELECT DISTINCT `{c}` FROM zomato_ai.gold.{t} WHERE `{c}` IS NOT NULL ORDER BY 1"
            ).collect()]
            value_lines.append(f"{t}.{c} values: {vals}")

value_text = "\n".join(value_lines)
print(value_text)

agg_city_order_status.order_status values: ['cancelled', 'delivered', 'refunded']
agg_restaurant_sentiment.top_complaint_topic values: ['delivery partner behavior', 'delivery speed', 'food quality', 'packaging', 'price and value']
dim_food.veg_or_non_veg values: ['non-veg', 'veg']
dim_users.gender values: ['female', 'male']
dim_users.marital_status values: ['Married', 'Prefer not to say', 'Single']
dim_users.occupation values: ['Employee', 'House wife', 'Self Employeed', 'Student']
dim_users.monthly_income values: ['10001 to 25000', '25001 to 50000', 'Below Rs.10000', 'More than 50000', 'No Income']
dim_users.educational_qualifications values: ['Graduate', 'Ph.D', 'Post Graduate', 'School', 'Uneducated']
event_log_a967f01d_ce2d_412e_a8cc_e927d8929b3b.level values: ['ERROR', 'INFO', 'METRICS', 'WARN']
event_log_a967f01d_ce2d_412e_a8cc_e927d8929b3b.maturity_level values: ['EVOLVING', 'STABLE']
event_log_a967f01d_ce2d_412e_a8cc_e927d8929b3b.event_type values: ['create_update', 'dataset_de

## 3. Build the instructions for the model
Combines the table descriptions, the allowed values, and the business rules (revenue counts delivered orders only, cancel rate, and so on) into one set of instructions.

In [0]:
rules = """
- Use only the tables listed above, always with full names like zomato_ai.gold.fact_orders.
- Write one Spark SQL SELECT statement. No explanations, no markdown, no semicolon.
- Use the exact allowed values listed above, including capitalization and spelling (even if they look like typos).
- When filtering on a text column, prefer lower(column) = 'value' in lowercase so capitalization can't cause a miss.
- Revenue means sum(sales_amount) over orders where order_status = 'delivered'.
- Cancel rate = cancelled orders / total orders.
- order_day_of_week: 1 = Sunday through 7 = Saturday.
- Restaurants with a null name, or named 'Unknown restaurant' in agg_restaurant_sentiment, have missing source data.
- Add LIMIT 20 unless the question asks for a specific number.
- For totals, rankings, and rates, prefer the agg_ tables. Use fact_ tables when the question needs a custom grouping or filter.
"""

system_prompt = f"""You write Spark SQL for Databricks.

Tables and columns:
{schema_text}

{value_text}

Rules:{rules}"""

print(system_prompt)

You write Spark SQL for Databricks.

Tables and columns:
zomato_ai.gold.agg_city_order_status: restaurant_city (STRING), order_status (STRING), orders (LONG), share_of_city_orders (DOUBLE)
zomato_ai.gold.agg_daily_sales: order_date (DATE), total_orders (LONG), delivered_orders (LONG), cancelled_orders (LONG), refunded_orders (LONG), revenue (DECIMAL), avg_delivery_time_min (DOUBLE), avg_customer_rating (DOUBLE)
zomato_ai.gold.agg_restaurant_performance: restaurant_id (STRING), restaurant_name (STRING), city (STRING), total_orders (LONG), delivered_orders (LONG), cancel_rate (DOUBLE), revenue (DECIMAL), avg_order_value (DECIMAL), avg_delivery_time_min (DOUBLE), avg_customer_rating (DOUBLE)
zomato_ai.gold.agg_restaurant_sentiment: restaurant_id (STRING), restaurant_name (STRING), city (STRING), total_reviews (LONG), avg_rating (DOUBLE), positive_reviews (LONG), neutral_reviews (LONG), mixed_reviews (LONG), negative_reviews (LONG), pct_negative (DOUBLE), top_complaint_topic (STRING), top_

## 4. The generator, the safety check, and the ask function
`generate_sql` sends the question to the model and gets SQL back. `check_sql` allows only a single SELECT on gold tables. `ask` runs the safe query and shows the SQL and the result, retrying once if the query errors. This is the only place the AI model is called.

In [0]:
import re
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()
BLOCKED = r"\b(insert|update|delete|drop|alter|create|truncate|merge|grant|revoke|replace|call|copy|optimize|vacuum|use|set)\b"

def generate_sql(question, feedback=None):
    messages = [ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt),
                ChatMessage(role=ChatMessageRole.USER, content=question)]
    if feedback:
        messages.append(ChatMessage(role=ChatMessageRole.USER, content=feedback))
    resp = w.serving_endpoints.query(name=MODEL, messages=messages,
                                     max_tokens=600, temperature=0)
    sql = resp.choices[0].message.content.strip()
    sql = re.sub(r"^```(?:sql)?|```$", "", sql, flags=re.MULTILINE).strip()
    return sql.rstrip(";").strip()

def check_sql(sql):
    s = sql.lower()
    if ";" in sql:
        raise ValueError("more than one statement")
    if not (s.startswith("select") or s.startswith("with")):
        raise ValueError("not a SELECT query")
    if re.search(BLOCKED, s):
        raise ValueError("contains a blocked keyword")
    if re.search(r"zomato_ai\.(bronze|silver)", s):
        raise ValueError("only gold tables are allowed")
    if "limit" not in s:
        sql += "\nLIMIT 100"
    return sql

def ask(question):
    print("Question:", question)
    feedback = None
    for attempt in range(2):
        sql = generate_sql(question, feedback)
        try:
            safe_sql = check_sql(sql)
        except ValueError as e:
            print(f"Blocked by guard ({e}):\n{sql}")
            return
        try:
            print("\nSQL used:\n" + safe_sql + "\n")
            display(spark.sql(safe_sql))
            return
        except Exception as e:
            feedback = f"That query failed with: {str(e)[:300]}\nQuery was:\n{sql}\nWrite a corrected query."
    print("Could not produce a working query.")

## 5. Test the safety check
Dangerous queries (a DROP, a hidden DELETE after a semicolon, a read from the silver layer) must be blocked, and a normal query must pass. No AI model is involved. A production version would also run queries as a user with read-only permission on the gold schema.

In [0]:
tests = [
    "DROP TABLE zomato_ai.gold.fact_orders",
    "SELECT * FROM zomato_ai.gold.fact_orders; DELETE FROM zomato_ai.gold.fact_orders",
    "SELECT * FROM zomato_ai.silver.users",
]

for bad in tests:
    try:
        check_sql(bad)
        print("NOT BLOCKED:", bad)
    except ValueError as e:
        print("blocked:", e, "|", bad)

# A safe query should pass, and get a LIMIT added
good = "SELECT order_status, count(*) FROM zomato_ai.gold.fact_orders GROUP BY order_status"
print("\npassed:\n" + check_sql(good))

blocked: not a SELECT query | DROP TABLE zomato_ai.gold.fact_orders
blocked: more than one statement | SELECT * FROM zomato_ai.gold.fact_orders; DELETE FROM zomato_ai.gold.fact_orders
blocked: only gold tables are allowed | SELECT * FROM zomato_ai.silver.users

passed:
SELECT order_status, count(*) FROM zomato_ai.gold.fact_orders GROUP BY order_status
LIMIT 100


## 6. Example questions
Each question prints the SQL the model wrote, then the result.

In [0]:
ask("What is the average age of self-employed users?")

Question: What is the average age of self-employed users?

SQL used:
SELECT AVG(age) FROM zomato_ai.gold.dim_users WHERE lower(occupation) = 'self employeed' LIMIT 20



AVG(age)
26.389276216488177


In [0]:
ask("Which city has the most cancelled orders?")

Question: Which city has the most cancelled orders?

SQL used:
SELECT restaurant_city, orders FROM zomato_ai.gold.agg_city_order_status WHERE lower(order_status) = 'cancelled' ORDER BY orders DESC LIMIT 1



restaurant_city,orders
Bikaner,7410


In [0]:
ask("What is the total revenue for each payment method?")

Question: What is the total revenue for each payment method?

SQL used:
SELECT payment_method, SUM(sales_amount) AS total_revenue FROM zomato_ai.gold.fact_orders WHERE lower(order_status) = 'delivered' GROUP BY payment_method LIMIT 20



payment_method,total_revenue
card,882536207.00
netbanking,221154782.00
wallet,663027454.00
cod,620247665.00
upi,2031203779.00


In [0]:
ask("Which 5 restaurants have the highest share of negative reviews, with at least 5 reviews?")

Question: Which 5 restaurants have the highest share of negative reviews, with at least 5 reviews?

SQL used:
SELECT restaurant_name, pct_negative FROM zomato_ai.gold.agg_restaurant_sentiment WHERE lower(restaurant_name) != 'unknown restaurant' AND total_reviews >= 5 ORDER BY pct_negative DESC LIMIT 5



restaurant_name,pct_negative
Shakes & Bakes,1.0
Kathleen,1.0
Royal Sandwich,1.0
London Dairy,1.0
GOBRGR@TRYST GOURMET,1.0
